In [55]:
## Create a vector of the required packages for this analysis
req_packages <- c("ape", "cowplot", "devtools", "EvoGeneX", "ggplot2", "ggpubr", 
                  "gridExtra", "janitor", "readxl", "stringr", "tidyverse", "vegan")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

In [2]:
## load in expression data
raw_reads <- read_csv("../obscura_group_total_reads.csv") %>%
    janitor::clean_names()

## load in tree
tree <- ape::read.tree("../select_obscura_tree.nwk")

Rows: 16880 Columns: 30
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): Gene_ID
dbl (29): Daff_1, Daff_2, Daff_3, Dazt_1, Dazt_2, Dazt_3, Dbif_1, Dbif_2, Db...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Create Regimes

In [3]:
## construct node annotations

## get the edge associated with each internal node
tree_edge <- tree$edge %>%
    as.data.frame() %>%
    rename(internal = 1, tip = 2)

## create a data frame for the tips
tip_label <- tree$tip.label %>%
    as.data.frame() %>%
    rename(node = 1) %>%
    rownames_to_column("tip") %>%
    mutate(tip = as.double(tip))
tip_df <- tip_label %>%
    select(-tip) %>%
    mutate(node2 = "")

## associate species with internal node
raw_tree_label <- tree_edge %>%
    left_join(tip_label, by = "tip")
head(raw_tree_label)

,internal,tip,node
,<int>,<dbl>,<chr>
1,9,1,Drosophila_bifasciata
2,9,10,NA
3,10,11,NA
4,11,2,Drosophila_obscura
5,11,12,NA
6,12,13,NA


In [4]:
## create a clean version of the tree to populate
tree_label <- raw_tree_label

## determine the parent species of the internal nodes
for (i in 1:nrow(raw_tree_label)) {

    if (is.na(tree_label[i, "node"])) {
        
        ## get the species associated with the parent node
        parent <- tree_label[i, "tip"]
        parent_species <- tree_label %>%
            filter(internal == parent) %>%
            pull("node")
        
        ## if the parent node has another parent
        if (is.na(parent_species[1])) {

            ## get the parent node of the parent
            parent_parent <- tree_label %>%
                filter(internal == parent) %>%
                pull("tip")
            parent_parent_species <- tree_label %>%
                filter(internal == parent_parent) %>%
                pull("node")
            
            ## if the parent's parent node has another parent
            if (is.na(parent_parent_species[1])) {
                ## get the parent node of the parent
                parent_parent_parent <- tree_label %>%
                    filter(internal == parent_parent) %>%
                    pull("tip")
                parent_parent_parent_species <- tree_label %>%
                    filter(internal == parent_parent_parent) %>%
                    pull("node")
                
                ## populate original with the species
                tree_label[i, "node"] <- parent_parent_parent_species[1]
            } else {

                ## populate original with the species
                tree_label[i, "node"] <- parent_parent_species[1]
            }
        } else {
            
            tree_label[i, "node"] <- parent_species[1]
        }
    }
}

In [5]:
## create base for all of the regimes
regime_base <- tree_label %>%
    arrange(internal) %>%
    mutate(tip = rep(c("node", "node2"), length.out = nrow(tree_label))) %>%
    pivot_wider(names_from = tip, values_from = node) %>%
    select(-internal) %>%
    bind_rows(tip_df)
regime_base

node,node2
<chr>,<chr>
Drosophila_bifasciata,Drosophila_guanche
Drosophila_obscura,Drosophila_pseudoobscura
Drosophila_obscura,Drosophila_subobscura
Drosophila_subobscura,Drosophila_guanche
Drosophila_subobscura,Drosophila_madeirensis
Drosophila_pseudoobscura,Drosophila_azteca
Drosophila_azteca,Drosophila_affinis
Drosophila_bifasciata,
Drosophila_obscura,


In [6]:
## create null regime
regime_null <- regime_base %>%
    cbind(regime = "empty")
head(regime_null)

,node,node2,regime
,<chr>,<chr>,<chr>
1,Drosophila_bifasciata,Drosophila_guanche,empty
2,Drosophila_obscura,Drosophila_pseudoobscura,empty
3,Drosophila_obscura,Drosophila_subobscura,empty
4,Drosophila_subobscura,Drosophila_guanche,empty
5,Drosophila_subobscura,Drosophila_madeirensis,empty
6,Drosophila_pseudoobscura,Drosophila_azteca,empty


In [7]:
## write parasperm regime to file
write_csv(regime_null, "regime_null.csv")

In [8]:
## create parasperm regime
regime_parasperm <- regime_base %>%
    cbind(regime = c("low", "low", "low", "medium", "medium", "medium", "high",
                     "high", "high", "low", "medium", "medium", "medium", "low", "medium"))
regime_parasperm

node,node2,regime
<chr>,<chr>,<chr>
Drosophila_bifasciata,Drosophila_guanche,low
Drosophila_obscura,Drosophila_pseudoobscura,low
Drosophila_obscura,Drosophila_subobscura,low
Drosophila_subobscura,Drosophila_guanche,medium
Drosophila_subobscura,Drosophila_madeirensis,medium
Drosophila_pseudoobscura,Drosophila_azteca,medium
Drosophila_azteca,Drosophila_affinis,high
Drosophila_bifasciata,,high
Drosophila_obscura,,high


In [9]:
## write parasperm regime to file
write_csv(regime_parasperm, "regime_parasperm.csv")

### Run EvoGeneX

In [16]:
## reformat the reads data frame
reads <- raw_reads %>%
    pivot_longer(!gene_id, names_to = "name", values_to = "value") %>%
    filter(!grepl("sr", name)) %>%
    separate_wider_delim(name, delim = "_", names = c("species", "replicate")) %>%
    mutate(species = case_when(species == "daff" ~ "Drosophila_affinis",
                               species == "dazt" ~ "Drosophila_azteca",
                               species == "dbif" ~ "Drosophila_bifasciata",
                               species == "dgua" ~ "Drosophila_guanche",
                               species == "dmad" ~ "Drosophila_madeirensis",
                               species == "dobs" ~ "Drosophila_obscura",
                               species == "dpse" ~ "Drosophila_pseudoobscura",
                               species == "dsub" ~ "Drosophila_subobscura"),
            replicate = paste0("rep", replicate)) %>%
    pivot_wider(names_from = replicate, values_from = value)
head(reads)

gene_id,species,rep1,rep2,rep3
<chr>,<chr>,<dbl>,<dbl>,<dbl>
LOC117184661,Drosophila_affinis,5,5,0
LOC117184661,Drosophila_azteca,0,0,0
LOC117184661,Drosophila_bifasciata,1,4,2
LOC117184661,Drosophila_guanche,0,0,0
LOC117184661,Drosophila_madeirensis,0,NA,0
LOC117184661,Drosophila_obscura,0,0,0


In [17]:
## get list of genes
genes <- unique(reads$gene_id)

## create empty data frame to populate
model_results <- matrix(data = NA, nrow = 1, ncol = 7, dimnames = list(c("test"),
                                                                       c("gene", "constraint_pvalue", "constraint_qvalue", 
                                                                        "adaptive_pvalue", "adaptive_qvalue", 
                                                                        "adaptive2_pvalue", "adaptive2_qvalue"))) %>%
    as.data.frame()

## create degrees of freedom for models
ou_dof <- (
  1   # alpha
  + 1 # sigma.sq
  + 1 # theta
  + 1 # gamma
)


ou2_dof <- (
  1   # alpha
  + 1 # sigma.sq
  + 2 # theta
  + 1 # gamma
)

brown_dof <- (
  1   # sigma.sq
  + 1 # theta
  + 1 # gamma
)


for (i in 1:length(genes)) {

    ## get gene name
    gene <- genes[i]

    gene_df <- reads %>%
      filter(gene_id == gene) %>%
      select(-gene_id)
    
    ## populated 2nd replicate of madeirensis with the average
    mad2 <- as.double(as.vector(gene_df[5,])) %>%
      na.omit() %>%
      sum()/2
    gene_df[5,3] <- mad2



    # Constraint versus Neutral

    ## fit replicated OU model
    evogene <- EvoGeneX()
    evogene$setTree("/home/clavery/projects/pse_ag/analysis/polyspermy/cagee_run/select_obscura_tree.nwk")
    evogene$setRegimes("regime_evogenex/regime_null.csv")
    ou_res <- evogene$fit(gene_df, alpha = 0.1, gamma = 0.01)

    ## fit replicated Brownian motion model
    brown <- Brown()
    brown$setTree("/home/clavery/projects/pse_ag/analysis/polyspermy/cagee_run/select_obscura_tree.nwk")
    brown_res <- brown$fit(gene_df, gamma = 0.01)

    ## calculate pvalue and qvalue
    ou_bm_pvalue <- 1 - pchisq((ou_res$loglik - brown_res$loglik) * 2, (ou_dof - brown_dof))
    ou_bm_qvalue <- p.adjust(ou_bm_pvalue, method = "fdr")


    # Adaptive versus Not

    ## fit replicated OU model
    evogene2 <- EvoGeneX()
    evogene2$setTree("/home/clavery/projects/pse_ag/analysis/polyspermy/cagee_run/select_obscura_tree.nwk")
    evogene2$setRegimes("regime_evogenex/regime_parasperm.csv")
    ou2_res <- evogene2$fit(gene_df, alpha = 0.1, gamma = 0.01)

    ## calculate pvalue and qvalue
    ou2_bm_pvalue <- 1 - pchisq((ou2_res$loglik - brown_res$loglik) * 2, (ou2_dof - brown_dof))
    ou2_bm_qvalue <- p.adjust(ou2_bm_pvalue, method = "fdr")

    ou2_ou_pvalue <- 1 - pchisq((ou2_res$loglik - ou_res$loglik) * 2, (ou2_dof - ou_dof))
    ou2_ou_qvalue <- p.adjust(ou2_ou_pvalue, method = "fdr")

    ## add values 
    model_results <- add_row(model_results, 
                             gene = gene,
                             constraint_pvalue = ou_bm_pvalue,
                             constraint_qvalue = ou_bm_qvalue,
                             adaptive_pvalue = ou2_bm_pvalue,
                             adaptive_qvalue = ou2_bm_qvalue,
                             adaptive2_pvalue = ou2_ou_pvalue,
                             adaptive2_qvalue = ou2_ou_qvalue)

}

## add labeling of constrained and adaptive genes
model_results <- model_results %>%
  mutate(constraint = case_when(constraint_qvalue < 0.05 ~ "constraint",
                                TRUE ~ "neutral"),
         adaptive = case_when(adaptive_qvalue < 0.05 & adaptive2_qvalue < 0.05 ~ "adaptive",
                              TRUE ~ "non-adaptive"))

## write data frame
write_csv(model_results, "evogenex_results.csv")

In [ ]:
# ## look for enriched GO terms among the genes associated with increased parasperm
# ## get lists to iterate through
#     ## genes
# genes <- unique(evogenex$gene)
#     ## ontology terms
# go_terms <- unique(ontology$term)

# ## create empty dataframe to populate
# go_compared <- data.frame(evolution = character(), 
#                           go_term = character(),
#                           go_genes = double(),
#                           go_ratio = double(),
#                           go_pvalue = double())

# for (k in unique(evogenex$constraint)) {
#     for (j in unique(evogenex$adaptive)) {

#         evolution_genes <- evogenex %>%
#             filter(constraint == k & adaptive == j) %>%
#             pull("gene")

#         for (i in go_terms) {    
                
#             ## format dataframe based on mutation type and associated go term
#             go_df <- ontology %>%
#                 mutate(evo = case_when(gene %in% evolution_genes ~ "evo",
#                                             TRUE ~ "nontype"),
#                        go = case_when(term == i ~ "go",
#                                       TRUE ~ "nogo")) %>%
#                 group_by(evo, go) %>%
#                 count() %>%
#                 pivot_wider(names_from = go, values_from = n) %>%
#                 column_to_rownames("evo") %>%
#                 mutate_all(replace_na, replace = 0)

#             ## get the number of genes with both mutation type and go term
#             evo_go <- go_df["evo", "go"]

#             ## only perform chi squared if there are more than 0 genes with go term
#             if (evo_go > 0) {        
                
#                 ## perform chi-squared test
#                 go_chi <- go_df %>%
#                     as.matrix() %>%
#                     chisq.test()
#                 go_pvalue <- go_chi$p.value

#                 ## populate dataframe with data
#                 go_compared <- add_row(go_compared, 
#                                             evolution = paste(k, j, sep = "_"),
#                                             go_term = i, 
#                                             go_genes = evo_go, 
#                                             go_ratio = evo_go/length(evolution_genes), 
#                                             go_pvalue = go_pvalue)
                
#             }    
#         }
#     }
# }

# write.csv(go_compared, "evogenex_go.csv")